## 1. Setup & Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics pandas pyyaml

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Dataset Setup & K-Fold (K=3) Splitting

In [ ]:
import os, zipfile, glob, random, yaml

zip_candidates = ['/content/drive/MyDrive/Trash (3).zip', '/content/drive/MyDrive/trash (3).zip']
zip_path = next((z for z in zip_candidates if os.path.exists(z)), None)
extract_path = '/content/trash_project'

if zip_path and not os.path.exists(os.path.join(extract_path, 'Trash_dataset_balanced')):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)

dataset_dir = None
for root, dirs, files in os.walk(extract_path):
    if os.path.basename(root) == 'Trash_dataset_balanced':
        dataset_dir = root
        break

assert dataset_dir is not None and os.path.exists(dataset_dir), 'Không tìm thấy thư mục Trash_dataset_balanced!'

# Quét sạch toàn bộ 26.048 ảnh từ cả 3 thư mục cũ (train, val, test) để gộp lại chia 3-Fold
all_imgs = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
    all_imgs.extend(glob.glob(f'{dataset_dir}/**/images/{ext}', recursive=True))
all_imgs = sorted(list(set(all_imgs)))
assert len(all_imgs) > 0, 'Không tìm thấy ảnh nào trong dataset!'
print(f'Tổng số ảnh thu thập được (gộp toàn bộ 26k ảnh): {len(all_imgs)}')

random.seed(42)
random.shuffle(all_imgs)

n = len(all_imgs)
f1, f2 = n // 3, 2 * (n // 3)
folds = [
    all_imgs[:f1],
    all_imgs[f1:f2],
    all_imgs[f2:]
]

class_names = ['battery', 'cardboard', 'paper', 'glass', 'metal', 'plastic', 'organic']

for k in range(1, 4):
    val_imgs = folds[k - 1]
    train_imgs = [img for i, f in enumerate(folds) if i != (k - 1) for img in f]
    
    train_txt = f'/content/train_fold{k}.txt'
    val_txt = f'/content/val_fold{k}.txt'
    with open(train_txt, 'w') as f:
        f.writelines(f'{p}\n' for p in train_imgs)
    with open(val_txt, 'w') as f:
        f.writelines(f'{p}\n' for p in val_imgs)
        
    yaml_dict = {
        'train': train_txt,
        'val': val_txt,
        'nc': 7,
        'names': {i: name for i, name in enumerate(class_names)}
    }
    with open(f'/content/data_fold{k}.yaml', 'w') as f:
        yaml.dump(yaml_dict, f, sort_keys=False)
    print(f'Fold {k}: Train={len(train_imgs)}, Val={len(val_imgs)} -> /content/data_fold{k}.yaml')


## 4. CBAM Module Definition & Registration

In [ ]:
import sys, inspect, torch
import torch.nn as nn
import ultralytics.nn.modules as modules
import ultralytics.nn.modules.block as block
import ultralytics.nn.tasks as tasks

class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        reduced_ch = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, reduced_ch, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced_ch, channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        scale = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv(scale))

class CBAM(nn.Module):
    def __init__(self, c1: int, c2: int = None, reduction: int = 16, kernel_size: int = 7):
        super().__init__()
        self.channel_attention = ChannelAttention(c1, reduction=reduction)
        self.spatial_attention = SpatialAttention(kernel_size=kernel_size)

    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))

setattr(modules, 'CBAM', CBAM)
setattr(block, 'CBAM', CBAM)
setattr(tasks, 'CBAM', CBAM)
tasks.__dict__['CBAM'] = CBAM
if '__main__' in sys.modules:
    setattr(sys.modules['__main__'], 'CBAM', CBAM)
if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([CBAM, ChannelAttention, SpatialAttention])

src = inspect.getsource(tasks.parse_model)
if 'CBAM,' not in src:
    src_patched = src.replace('C2PSA,', 'C2PSA, CBAM,')
    exec_globals = tasks.__dict__
    exec(src_patched, exec_globals)
    tasks.parse_model = exec_globals['parse_model']

## 5. Model Architecture

In [ ]:
yaml_arch = """nc: 7
scales:
  s: [0.50, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 1, CBAM, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]
  - [[16, 19, 22], 1, Detect, [nc]]
"""

os.makedirs('/content/models', exist_ok=True)
with open('/content/models/yolo11s-cbam.yaml', 'w') as f:
    f.write(yaml_arch.strip())

## 6. Train Model (Chọn FOLD = 1, 2 hoặc 3)

In [ ]:
import shutil
from ultralytics import YOLO

# CHỌN FOLD HUẤN LUYỆN (Tài khoản 1 đặt 1, tài khoản 2 đặt 2, tài khoản 3 đặt 3)
FOLD = 1

data_yaml = f'/content/data_fold{FOLD}.yaml'
exp_name = f'trash_yolo11s_cbam_fold{FOLD}'

model = YOLO('/content/models/yolo11s-cbam.yaml')
model.load('yolo11s.pt')

local_project = '/content/runs/detect'
local_save_dir = f'{local_project}/{exp_name}'
drive_save_dir = f'/content/drive/MyDrive/Trash_YOLO11_Balanced_Runs/{exp_name}'
os.makedirs(drive_save_dir, exist_ok=True)
os.makedirs(f'{drive_save_dir}/weights', exist_ok=True)

def safe_sync(src, dst):
    if not os.path.exists(src):
        return
    try:
        if os.path.exists(dst):
            os.remove(dst)
    except Exception:
        pass
    try:
        shutil.copy2(src, dst)
    except Exception:
        pass

def on_fit_epoch_end(trainer):
    if hasattr(trainer, 'stopper') and trainer.stopper is not None:
        delta = trainer.epoch - trainer.stopper.best_epoch
        if delta == 5 and not getattr(trainer, '_lr_reduced_on_plateau', False):
            trainer._lr_reduced_on_plateau = True
            for param_group in trainer.optimizer.param_groups:
                param_group['lr'] *= 0.5
            if hasattr(trainer, 'scheduler') and hasattr(trainer.scheduler, 'base_lrs'):
                trainer.scheduler.base_lrs = [b * 0.5 for b in trainer.scheduler.base_lrs]
            lr = trainer.optimizer.param_groups[0]['lr']
            print(f"[ReduceLROnPlateau] Reducing LR to: {lr:.6f}")
        elif delta < 5:
            trainer._lr_reduced_on_plateau = False

    for fname in ['results.csv', 'args.yaml']:
        safe_sync(f'{local_save_dir}/{fname}', f'{drive_save_dir}/{fname}')
    for wname in ['last.pt', 'best.pt']:
        safe_sync(f'{local_save_dir}/weights/{wname}', f'{drive_save_dir}/weights/{wname}')

model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

model.train(
    data=data_yaml,
    epochs=100,
    batch=32,
    imgsz=640,
    device=0,
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    patience=10,
    cos_lr=False,
    weight_decay=0.0005,
    dropout=0.1,
    mixup=0.15,
    copy_paste=0.3,
    erasing=0.4,
    project=local_project,
    name=exp_name,
    exist_ok=True,
    save=True,
    verbose=True
)

if os.path.exists(local_save_dir):
    shutil.copytree(local_save_dir, drive_save_dir, dirs_exist_ok=True)

## 7. Resume (Chọn đúng FOLD cần tiếp tục)

In [ ]:
import os, shutil, glob
import pandas as pd
from ultralytics import YOLO

# CHỌN ĐÚNG FOLD ĐANG RESUME (1, 2 hoặc 3)
FOLD = 3
exp_name = f'trash_yolo11s_cbam_fold{FOLD}'

candidates = [
    f'/content/drive/MyDrive/{exp_name}',
    f'/content/drive/MyDrive/Trash_YOLO11_Balanced_Runs/{exp_name}'
]
drive_exp = next((c for c in candidates if os.path.exists(os.path.join(c, 'weights', 'last.pt'))), candidates[0])
local_exp = f'/content/runs/detect/{exp_name}'

last_pt_drive = f'{drive_exp}/weights/last.pt'
assert os.path.exists(last_pt_drive), f'Không tìm thấy file checkpoint: {last_pt_drive}'
print(f'-> ĐÃ TÌM THẤY CHECKPOINT TẠI: {last_pt_drive}')

os.makedirs(f'{local_exp}/weights', exist_ok=True)
shutil.copy2(last_pt_drive, f'{local_exp}/weights/last.pt')

if os.path.exists(f'{drive_exp}/args.yaml'):
    shutil.copy2(f'{drive_exp}/args.yaml', f'{local_exp}/args.yaml')

csvs = glob.glob(f'{drive_exp}/results*.csv')
if csvs:
    dfs = []
    for f in csvs:
        if os.path.getsize(f) > 0:
            d = pd.read_csv(f)
            d.columns = d.columns.str.strip()
            dfs.append(d)
    if dfs:
        merged = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=['epoch']).sort_values('epoch')
        merged.to_csv(f'{local_exp}/results.csv', index=False)

def safe_sync(src, dst):
    if not os.path.exists(src):
        return
    try:
        if os.path.exists(dst):
            os.remove(dst)
    except Exception:
        pass
    try:
        shutil.copy2(src, dst)
    except Exception:
        pass

def on_fit_epoch_end(trainer):
    if hasattr(trainer, 'stopper') and trainer.stopper is not None:
        delta = trainer.epoch - trainer.stopper.best_epoch
        if delta == 5 and not getattr(trainer, '_lr_reduced_on_plateau', False):
            trainer._lr_reduced_on_plateau = True
            for param_group in trainer.optimizer.param_groups:
                param_group['lr'] *= 0.5
            if hasattr(trainer, 'scheduler') and hasattr(trainer.scheduler, 'base_lrs'):
                trainer.scheduler.base_lrs = [b * 0.5 for b in trainer.scheduler.base_lrs]
            lr = trainer.optimizer.param_groups[0]['lr']
            print(f"[ReduceLROnPlateau] Reducing LR to: {lr:.6f}")
        elif delta < 5:
            trainer._lr_reduced_on_plateau = False

    for fname in ['results.csv', 'args.yaml']:
        safe_sync(f'{local_exp}/{fname}', f'{drive_exp}/{fname}')
    for wname in ['last.pt', 'best.pt']:
        safe_sync(f'{local_exp}/weights/{wname}', f'{drive_exp}/weights/{wname}')

resume_model = YOLO(f'{local_exp}/weights/last.pt')
resume_model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
resume_model.train(resume=True)

if os.path.exists(local_exp):
    shutil.copytree(local_exp, drive_exp, dirs_exist_ok=True)

## 8. Đánh giá Fold (Validation trên Fold được giữ lại)


In [ ]:
from ultralytics import YOLO

FOLD = 1
exp_name = f'trash_yolo11s_cbam_fold{FOLD}'

best_candidates = [
    f'/content/runs/detect/{exp_name}/weights/best.pt',
    f'/content/drive/MyDrive/Trash_YOLO11_Balanced_Runs/{exp_name}/weights/best.pt'
]
best_pt = next((b for b in best_candidates if os.path.exists(b)), None)
assert best_pt is not None, f'Không tìm thấy best.pt cho Fold {FOLD}!'

eval_model = YOLO(best_pt)
test_res = eval_model.val(
    data=f'/content/data_fold{FOLD}.yaml',
    split='val',
    imgsz=640,
    device=0,
    verbose=True
)

print(f"Precision: {test_res.box.mp:.4f}")
print(f"Recall: {test_res.box.mr:.4f}")
print(f"mAP50: {test_res.box.map50:.4f}")
print(f"mAP50-95: {test_res.box.map:.4f}")
print(f"Latency: {test_res.speed.get('inference', 0.0):.2f} ms")
for i, c in enumerate(test_res.names.values()):
    print(f"  {c:<12}: {test_res.box.maps[i]:.4f}")